# Halifax Transit Ridership — Data Cleaning & Exploration

**Goal:** turn Halifax Transit's raw automatic passenger count (APC) data into a clean,
analysis-ready dataset, and check that the data is trustworthy before any analysis.

**Input:** `data/apc_ridership.csv` — one row per route, direction and half-hour slot per day
**Output:** `data/apc_clean.csv` — outliers removed, dates parsed, `Weekday` and `Day_Type` added

| Section | What it does |
|---|---|
| 1 | Load and inspect the raw data |
| 2 | Find and remove sensor outliers |
| 3 | Parse dates and add a weekday column |
| 4 | Check date coverage (missing days, short days) |
| 5 | Classify each day as Weekday / Saturday / Sunday-Holiday |
| 6 | First look at ridership patterns |
| 7 | Save the clean dataset |

## 1. Load and inspect the raw data

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")

df = pd.read_csv(DATA_DIR / "apc_ridership.csv")
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")

Rows: 863,614   Columns: 8


In [2]:
# Column types and a preview of the first rows
print(df.dtypes)
df.head()

OBJECTID                    int64
Route_Number                  str
Route_Name                    str
Route_Direction               str
Ridership_Total             int64
Route_Hour                float64
Route_Hour_Description        str
Route_Date                    str
dtype: object


,OBJECTID,Route_Number,Route_Name,Route_Direction,Ridership_Total,Route_Hour,Route_Hour_Description,Route_Date
0,1,320,Airport - Fall River Regional Exp,Oud,8,4.5,4:30am to 4:59am,10/22/2026 10:00:00 AM
1,2,10C,Dalhousie,Ind,19,5.0,5:00am to 5:29am,10/22/2026 10:00:00 AM
2,3,10C,Dalhousie,Oud,3,5.0,5:00am to 5:29am,10/22/2026 10:00:00 AM
3,4,182,First Lake Express,Ind,23,5.0,5:00am to 5:29am,10/22/2026 10:00:00 AM
4,5,21,Lakeside - Timberlea,Oud,2,5.0,5:00am to 5:29am,10/22/2026 10:00:00 AM


Each row is the number of riders counted on **one route, in one direction, in one half-hour slot,
on one date**. `Route_Hour` is the slot start as a decimal (e.g. `15.5` = 3:30pm).

## 2. Find and remove sensor outliers

In [3]:
# Overall average vs. the single largest value
print(f"Mean riders per slot: {df['Ridership_Total'].mean():.1f}")
print(f"Max riders per slot:  {df['Ridership_Total'].max():,}")

df.sort_values("Ridership_Total", ascending=False).head(10)

Mean riders per slot: 26.8
Max riders per slot:  3,930


,OBJECTID,Route_Number,Route_Name,Route_Direction,Ridership_Total,Route_Hour,Route_Hour_Description,Route_Date
158727,158728,87,Sackville-Dartmouth,Ind,3930,12.5,12:30pm to 12:59pm,1/16/2026 10:00:00 AM
526257,526258,1,Spring Garden,Ind,3358,16.0,4:00pm to 4:29pm,8/6/2026 10:00:00 AM
172558,172559,5,Portland,Ind,2800,11.0,11:00am to 11:29am,1/16/2026 10:00:00 AM
146960,146961,84,Glendale Express,Oud,2449,11.5,11:30am to 11:59am,1/16/2026 10:00:00 AM
675415,675416,1,Spring Garden,Oud,2378,13.5,1:30pm to 1:59pm,8/6/2026 10:00:00 AM
172467,172468,5,Portland,Oud,2359,10.0,10:00am to 10:29am,1/16/2026 10:00:00 AM
390338,390339,29,Barrington,Oud,2137,16.0,4:00pm to 4:29pm,5/14/2026 10:00:00 AM
172386,172387,4,Universities,Ind,1855,9.0,9:00am to 9:29am,1/16/2026 10:00:00 AM
425125,425126,29,Barrington,Ind,1836,15.0,3:00pm to 3:29pm,5/14/2026 10:00:00 AM
155815,155816,4,Universities,Oud,1793,8.5,8:30am to 8:59am,1/16/2026 10:00:00 AM


In [4]:
# Is the maximum realistic? Compare it with the normal value for the same route and time slot.
top = df.loc[df["Ridership_Total"].idxmax()]
same_slot = df[(df["Route_Number"] == top["Route_Number"]) & (df["Route_Hour"] == top["Route_Hour"])]

print(f"Route {top['Route_Number']} at {top['Route_Hour_Description']}:")
print(f"  usual average = {same_slot['Ridership_Total'].mean():.0f} riders, max recorded = {top['Ridership_Total']:,}")

Route 87 at 12:30pm to 12:59pm:
  usual average = 31 riders, max recorded = 3,930


A single bus cannot carry ~3,900 people in 30 minutes, and the same route and slot normally
averages about 31 riders. Values like this are **counting errors, not real crowds**.

Only a handful of rows are above 300, so 300 is used as a conservative ceiling for a single
route-direction in one half-hour.

In [5]:
OUTLIER_THRESHOLD = 300

n_outliers = (df["Ridership_Total"] > OUTLIER_THRESHOLD).sum()
print(f"Rows above {OUTLIER_THRESHOLD}: {n_outliers} ({n_outliers / len(df):.4%} of the data)")

# .copy() makes df_clean an independent table, avoiding SettingWithCopyWarning later
df_clean = df.loc[df["Ridership_Total"] <= OUTLIER_THRESHOLD].copy()
print(f"Rows kept: {len(df_clean):,}")

Rows above 300: 64 (0.0074% of the data)
Rows kept: 863,550


## 3. Parse dates and add a weekday column

In [6]:
# Raw dates look like '10/22/2026 10:00:00 AM'; the time part carries no information
df_clean["Route_Date"] = pd.to_datetime(df_clean["Route_Date"], format="%m/%d/%Y %I:%M:%S %p")
df_clean["Route_Date"] = df_clean["Route_Date"].dt.normalize()   # keep the date only
df_clean["Weekday"] = df_clean["Route_Date"].dt.day_name()

df_clean[["Route_Date", "Weekday"]].head()

,Route_Date,Weekday
0,2026-10-22,Thursday
1,2026-10-22,Thursday
2,2026-10-22,Thursday
3,2026-10-22,Thursday
4,2026-10-22,Thursday


**Note:** `Route_Hour` values from 24.0 to 25.5 are not errors. They are after-midnight trips that
belong to the previous day's service (e.g. 24.5 = 12:30am), a standard transit-scheduling convention.

## 4. Check date coverage

Before comparing days, check that the data is complete and balanced. Otherwise a weekday could look
quiet simply because the data has fewer of them.

In [7]:
print(f"Date range: {df_clean['Route_Date'].min():%Y-%m-%d} to {df_clean['Route_Date'].max():%Y-%m-%d}")
print(f"Distinct dates: {df_clean['Route_Date'].nunique()}")

# Which calendar days are missing completely?
all_days = pd.date_range(df_clean["Route_Date"].min(), df_clean["Route_Date"].max(), freq="D")
missing = all_days.difference(df_clean["Route_Date"].unique())
print("Missing date(s):", [f"{d:%Y-%m-%d} ({d:%A})" for d in missing])

Date range: 2026-01-01 to 2026-10-27
Distinct dates: 299
Missing date(s): ['2026-09-19 (Saturday)']


In [8]:
# How many dates does each weekday have? (Balanced coverage = fair comparisons)
df_clean.groupby("Weekday")["Route_Date"].nunique()

Weekday
Friday       43
Monday       43
Saturday     42
Sunday       43
Thursday     43
Tuesday      43
Wednesday    42
Name: Route_Date, dtype: int64

In [9]:
# Rows per date: days with far fewer rows than usual are either reduced-service days or incomplete
rows_per_date = (
    df_clean.groupby(["Route_Date", "Weekday"])
    .size()
    .reset_index(name="rows")
)
rows_per_date.groupby("Weekday")["rows"].mean().round(0)

Weekday
Friday       3161.0
Monday       3056.0
Saturday     2463.0
Sunday       2053.0
Thursday     3160.0
Tuesday      3187.0
Wednesday    3133.0
Name: rows, dtype: float64

Weekends have fewer rows because fewer routes run. Any **Monday–Friday** date with a weekend-sized
row count is likely a holiday running the reduced schedule — check that next.

## 5. Classify each day: Weekday / Saturday / Sunday-Holiday

In [10]:
# Mon-Fri dates with a reduced (Sunday-sized) schedule
WEEKDAY_FULL_SERVICE_MIN_ROWS = 3000

is_mon_fri = ~rows_per_date["Weekday"].isin(["Saturday", "Sunday"])
reduced = rows_per_date["rows"] < WEEKDAY_FULL_SERVICE_MIN_ROWS
holiday_days = rows_per_date[is_mon_fri & reduced]

holiday_days

,Route_Date,Weekday,rows
0,2026-01-01,Thursday,2053
46,2026-02-16,Monday,2053
92,2026-04-03,Friday,2053
137,2026-05-18,Monday,2053
181,2026-07-01,Wednesday,2053
214,2026-08-03,Monday,2053
249,2026-09-07,Monday,2053
271,2026-09-30,Wednesday,2053
283,2026-10-12,Monday,2053


Every one of these dates is a Nova Scotia holiday (New Year's Day, Heritage Day, Good Friday,
Victoria Day, Canada Day, Natal Day, Labour Day, Truth and Reconciliation Day, Thanksgiving),
and each has exactly the same row count as a typical Sunday — **Halifax Transit runs its Sunday
schedule on holidays.** They are grouped with Sundays.

In [11]:
is_holiday = df_clean["Route_Date"].isin(holiday_days["Route_Date"])

# np.select is a fast, vectorised if / elif / else over the whole column
df_clean["Day_Type"] = np.select(
    [is_holiday | (df_clean["Weekday"] == "Sunday"),
     df_clean["Weekday"] == "Saturday"],
    ["Sunday/Holiday", "Saturday"],
    default="Weekday",
)

df_clean["Day_Type"].value_counts()

Day_Type
Weekday           653356
Sunday/Holiday    106756
Saturday          103438
Name: count, dtype: int64

## 6. First look at ridership patterns

To compare day types fairly, first total riders **per day**, then average those daily totals.

In [12]:
daily = (
    df_clean.groupby(["Route_Date", "Day_Type"])["Ridership_Total"]
    .sum()
    .reset_index()
)
avg_by_day_type = daily.groupby("Day_Type")["Ridership_Total"].mean().round(0)
print(avg_by_day_type)

ratio = avg_by_day_type["Weekday"] / avg_by_day_type["Sunday/Holiday"]
print(f"\nA typical weekday carries {ratio:.1f}x the riders of a Sunday/Holiday.")

Day_Type
Saturday          54252.0
Sunday/Holiday    39805.0
Weekday           91434.0
Name: Ridership_Total, dtype: float64

A typical weekday carries 2.3x the riders of a Sunday/Holiday.


In [13]:
# Busiest and quietest routes (total riders, Jan-Oct 2026)
route_totals = df_clean.groupby("Route_Name")["Ridership_Total"].sum().sort_values(ascending=False)

print("Busiest routes:")
print(route_totals.head(5).to_string())
print("\nQuietest routes:")
print(route_totals.tail(5).to_string())

Busiest routes:
Route_Name
Spring Garden         1994300
CrossTown             1761818
Fairview              1326157
Sackville Corridor    1200121
Peninsula             1191182

Quietest routes:
Route_Name
Porters Lake Regional Express    23064
Tantallon                        22151
Dockyard                         16059
Purcells Cove                    10450
Springvale                       10321


In [14]:
# Busiest half-hour slots on weekdays (average riders per weekday)
weekday_slots = (
    df_clean[df_clean["Day_Type"] == "Weekday"]
    .groupby("Route_Hour_Description")["Ridership_Total"]
    .sum()
    / (daily["Day_Type"] == "Weekday").sum()
)
weekday_slots.sort_values(ascending=False).head(5).round(0)

Route_Hour_Description
3:30pm to 3:59pm    4928.0
3:00pm to 3:29pm    4828.0
4:00pm to 4:29pm    4754.0
4:30pm to 4:59pm    4252.0
7:30am to 7:59am    4176.0
Name: Ridership_Total, dtype: float64

Deeper analysis (route rankings, rush-hour timing, weekend retention, monthly trend) is done in SQL — see the `sql/` folder.

## 7. Save the clean dataset

In [15]:
df_clean.to_csv(DATA_DIR / "apc_clean.csv", index=False)

# Quick check that the saved file has every column, including Day_Type
print(pd.read_csv(DATA_DIR / "apc_clean.csv", nrows=3).columns.tolist())
print(f"Saved {len(df_clean):,} rows to data/apc_clean.csv")

['OBJECTID', 'Route_Number', 'Route_Name', 'Route_Direction', 'Ridership_Total', 'Route_Hour', 'Route_Hour_Description', 'Route_Date', 'Weekday', 'Day_Type']
Saved 863,550 rows to data/apc_clean.csv
